# Notebook 1: Universe Screening and Pair Selection

This notebook documents the pair selection funnel:
- Load ASX 50 price data and verify coverage
- Apply the full cointegration test battery (in-sample only)
- Show how many pairs survive each filter
- Report the top 5 pairs selected for backtesting

**Critical**: pair selection uses only 2015–2020 in-sample data.
Out-of-sample (2021–present) is loaded but never touched until Notebook 3.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.data_loader import load_prices, split_in_out_of_sample, get_universe_by_sector, TICKER_SECTOR
from src.pair_selection import select_pairs, funnel_dataframe

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

## 1. Load price data

In [ ]:
prices = load_prices()
print(f"Price data shape: {prices.shape}")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"\nTickers loaded: {list(prices.columns)}")

In [ ]:
# Check coverage: what fraction of dates has data for each ticker?
coverage = (prices.notna().sum() / len(prices)).sort_values(ascending=False)
print("Coverage (fraction of dates with non-NaN price):")
print(coverage.to_string())

In [ ]:
# Split into in-sample (2015–2020) and out-of-sample (2021–present)
# Out-of-sample is loaded here but will not be used until Notebook 3.
prices_is, prices_oos = split_in_out_of_sample(prices, in_sample_end='2020-12-31')

print(f"In-sample:      {prices_is.index[0].date()} to {prices_is.index[-1].date()}  ({len(prices_is)} days)")
print(f"Out-of-sample:  {prices_oos.index[0].date()} to {prices_oos.index[-1].date()}  ({len(prices_oos)} days)")

## 2. Universe overview

The ASX 50 universe is grouped by GICS sector. We only test pairs within the same sector to reduce spurious cointegration.

In [ ]:
universe = get_universe_by_sector()
available_tickers = set(prices_is.columns)

print(f"{'Sector':<25} {'Defined':<10} {'Available'}")
print('-' * 50)
for sector, tickers in universe.items():
    avail = [t for t in tickers if t in available_tickers]
    print(f"{sector:<25} {len(tickers):<10} {len(avail)}  {avail}")

In [ ]:
# Plot normalised price histories for two sectors to give visual context
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, sector in zip(axes, ['Financials', 'Materials']):
    tickers = [t for t in universe[sector] if t in prices_is.columns]
    if not tickers:
        continue
    # Normalise to 100 at first observation
    sub = prices_is[tickers].dropna(how='all')
    normed = sub.div(sub.iloc[0]) * 100
    normed.plot(ax=ax, lw=1.2, legend=True)
    ax.set_title(f'{sector} — normalised prices (in-sample)', fontsize=11)
    ax.set_ylabel('Price (rebased to 100)')
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Pair selection funnel

We run the full four-gate pipeline on **in-sample data only**.

| Gate | Test | Criterion |
|------|------|-----------|
| 1 | ADF unit root on each series | Both must be I(1) |
| 2 | Engle-Granger cointegration | EG p-value ≤ 0.05 |
| 3 | Johansen trace test | Trace stat > 95% critical value |
| 4 | OU half-life | 1 ≤ half-life ≤ 30 trading days |

In [ ]:
# This is the only call that examines in-sample data for pair selection.
# prices_oos is never passed to select_pairs.
selected_pairs, funnel, all_analyses = select_pairs(
    prices_insample=prices_is,
    top_n=5,
    verbose=True,
)

In [ ]:
# Display the funnel summary
print(funnel.summary())

In [ ]:
# Funnel bar chart
stages = [
    ('Same-sector', funnel.stage_0_sector_pairs),
    ('Both I(1)', funnel.stage_1_both_i1),
    ('Engle-Granger', funnel.stage_2_eg_pass),
    ('Johansen', funnel.stage_3_johansen_pass),
    ('Half-life', funnel.stage_4_halflife_pass),
    ('Selected', funnel.selected),
]

fig, ax = plt.subplots(figsize=(10, 4))
labels, counts = zip(*stages)
bars = ax.bar(labels, counts, color=['steelblue']*5 + ['darkorange'], edgecolor='black', linewidth=0.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(count),
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Pair Selection Funnel — ASX 50 In-Sample (2015–2020)', fontsize=12)
ax.set_ylabel('Number of Pairs')
ax.set_ylim(0, max(counts) * 1.2)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Full results table — all pairs tested with rejection reasons
df_funnel = funnel_dataframe(all_analyses)
print(f"Total pairs tested: {len(df_funnel)}")
print(f"Passed all gates: {df_funnel['passed'].sum()}")
print()
df_funnel.sort_values(['passed', 'eg_p_value'], ascending=[False, True]).head(20)

## 4. Selected pairs

The top 5 pairs ranked by Engle-Granger p-value (then half-life):

In [ ]:
print(f"{'Rank':<5} {'Pair':<25} {'Sector':<25} {'EG p-value':<14} {'Half-life (d)':<16} {'Beta'}")
print('-' * 90)
for sp in selected_pairs:
    y, x = sp.analysis.ticker_y, sp.analysis.ticker_x
    sector = TICKER_SECTOR.get(y, '?')
    print(f"#{sp.rank:<4} {y}/{x:<23} {sector:<25} {sp.eg_p_value:<14.4f} {sp.half_life_days:<16.1f} {sp.ols_beta:.3f}")

In [ ]:
# Plot in-sample normalised spread for each selected pair
fig, axes = plt.subplots(len(selected_pairs), 1, figsize=(13, 3 * len(selected_pairs)), sharex=False)
if len(selected_pairs) == 1:
    axes = [axes]

for ax, sp in zip(axes, selected_pairs):
    y = prices_is[sp.analysis.ticker_y].dropna()
    x = prices_is[sp.analysis.ticker_x].dropna()
    common = y.index.intersection(x.index)
    spread = (y.loc[common] - sp.ols_beta * x.loc[common])
    spread_z = (spread - spread.mean()) / spread.std()
    ax.plot(spread_z.index, spread_z.values, lw=0.9, color='steelblue')
    ax.axhline(0, color='black', lw=0.8, linestyle='--')
    ax.axhline(2, color='crimson', lw=0.6, linestyle=':', alpha=0.7)
    ax.axhline(-2, color='crimson', lw=0.6, linestyle=':', alpha=0.7)
    ax.set_title(f"#{sp.rank} {sp.analysis.ticker_y}/{sp.analysis.ticker_x}  "
                 f"EG p={sp.eg_p_value:.4f}  HL={sp.half_life_days:.1f}d", fontsize=10)
    ax.set_ylabel('Z-score')
    ax.grid(True, alpha=0.3)

plt.suptitle('In-Sample Spread Z-Scores (static beta)', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Save selected pairs for use in subsequent notebooks
import pickle, os
os.makedirs('../data', exist_ok=True)
with open('../data/selected_pairs.pkl', 'wb') as f:
    pickle.dump(selected_pairs, f)
print("Selected pairs saved to ../data/selected_pairs.pkl")